# Load data

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from walinet.parameter_calibration.load_data import *

from walinet.parameter_calibration.compute_statistics import *

from walinet.parameter_calibration.pipeline_FWHM_SNR_shifts import *

from walinet.parameter_calibration.water_lipid_ratios import *

from walinet.parameter_calibration.plot_statistics import *

In [ ]:
bandwidth_hz = 2778.0
nmr_frequency_hz = 297_222_931.0
water_ppm = 4.68

In [ ]:
# SUBJECT_DIRS = [
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol03_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol04_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol05_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Brisbane/Vol07_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol01_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol02_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol03_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol04_Dat_NoL2_GradDel",
#     "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/London/Vol05_Dat_NoL2_GradDel"
# ]

# Load water, lipid and metabo data

In [ ]:
# Load water and mask

# water_by_subject, mask_by_subject = (
#     load_subject_water_fids(
#         base_path="/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/Denoising/datasets/Proton/7T/NoB0Correction",
#         subject_folders=[
#             "Vol1_Brisbane",
#         ],
#     )
# )

Water, brain_mask = load_subject_reconstructed_water_fids(
    base_path="/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/Denoising/datasets/Proton/7T/NoB0Correction",
    subject_folders=[
        "Vol1_Brisbane", "Vol3_Brisbane", "Vol4_Brisbane", "Vol5_Brisbane", "Vol6_Brisbane",
        "Vol1_London", "Vol2_London",  "Vol3_London", "Vol4_London" 
    ],
)

# Load full FIDs and FIDs with WALINET predicted nuisance subtracted ~ metabos

(
    original_fids_by_subject,
    after_walinet_fids_by_subject,
) = load_subject_original_and_walinet_fids(
    base_path="/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/7T/B0corrected_wo_LipidMask",
    subject_folders=[
        "Vol1_Brisbane", "Vol3_Brisbane", "Vol4_Brisbane", "Vol5_Brisbane", "Vol6_Brisbane",
        "Vol1_London", "Vol2_London",  "Vol3_London", "Vol4_London" 
    ],
)

In [ ]:
Water = Water
brain_mask = brain_mask

FullData = original_fids_by_subject
Metabos = after_walinet_fids_by_subject

Baseline = FullData - Metabos
Lipids = Baseline - Water

# Compute Lipd / Metabo, Water / Metabo tensor ratios

In [ ]:
n_points = Water.shape[-2]

frequency_hz = np.fft.fftshift(
    np.fft.fftfreq(
        n_points,
        d=1.0 / bandwidth_hz,
    )
)

ppm = (
    water_ppm
    + frequency_hz
    / (nmr_frequency_hz / 1e6)
)

water_ratio, lipid_ratio = calculate_component_ratios(
    Water,
    Lipids,
    Metabos,
    brain_mask,
    ppm,
    lipid_ppm_max=4.0,
)

print(water_ratio.shape)
print(lipid_ratio.shape)
# (X, Y, Z, S)

water_ratio_pool = pool_valid_voxels(
    water_ratio,
    brain_mask,
)

lipid_ratio_pool = pool_valid_voxels(
    lipid_ratio,
    brain_mask,
)

print(water_ratio_pool.shape)
print(lipid_ratio_pool.shape)

In [ ]:
water_median, water_iqr = calculate_pooled_median_iqr(
    water_ratio_pool
)

lipid_median, lipid_iqr = calculate_pooled_median_iqr(
    lipid_ratio_pool
)

print(
    f"Water / Metabolites: median={water_median:.3f}, "
    f"IQR={water_iqr:.3f}"
)

print(
    f"Lipids / Metabolites: median={lipid_median:.3f}, "
    f"IQR={lipid_iqr:.3f}"
)


In [ ]:
water_model = compare_positive_models(
    water_ratio_pool,
    title="Water / Metabolites",
    xlabel="Water / Metabolite ratio",
    bins=80,
    truncated_normal_sigma_factor=2.0,
    lognormal_sigma_factor=2.0,
    plot_percentile=99.5,
)

lipid_model = compare_positive_models(
    lipid_ratio_pool,
    title="Lipid / Metabolites",
    xlabel="Lipid / Metabolite ratio",
    bins=80,
    truncated_normal_sigma_factor=2.0,
    lognormal_sigma_factor=2.0,
    plot_percentile=99.5,
)